In [ ]:
# Example 1: Version info.
import apsw
from packaging.version import Version

print(f"apsw version:        {apsw.apsw_version()}")
print(f"sqlite lib version:  {apsw.sqlite_lib_version()}")
print(f"sqlite header ver:   {apsw.SQLITE_VERSION_NUMBER}")

# The SQLite library version is a plain dotted release number.
sqlite_version = Version(apsw.sqlite_lib_version())
print(f"\nparsed SQLite version: {sqlite_version}")

In [ ]:
# Example 2: Basic Connection + Cursor against an in-memory DB.
import apsw

conn = apsw.Connection(':memory:')
cur = conn.cursor()

cur.execute('CREATE TABLE items (id INTEGER PRIMARY KEY, name TEXT, qty INTEGER)')
cur.executemany(
    'INSERT INTO items (name, qty) VALUES (?, ?)',
    [('apple', 3), ('banana', 7), ('cherry', 12)],
)

rows = list(cur.execute('SELECT name, qty FROM items ORDER BY qty'))
print(rows)
assert rows == [('apple', 3), ('banana', 7), ('cherry', 12)]

(total,) = next(cur.execute('SELECT SUM(qty) FROM items'))
assert total == 22
conn.close()


In [ ]:
# Example 3: Compile options and Cursor.description.
# description and description_full are only defined WHILE a statement is executing:
# after the result set is exhausted apsw raises ExecutionCompleteError by design, so
# capture them mid-iteration over a non-empty result. description_full also needs
# SQLite built with SQLITE_ENABLE_COLUMN_METADATA, so it is checked only when present.
import apsw

conn = apsw.Connection(':memory:')
cur = conn.cursor()
options = {r[0] for r in cur.execute('PRAGMA compile_options')}
print(f"SQLite reports {len(options)} compile options")
has_column_metadata = 'ENABLE_COLUMN_METADATA' in options

cur.execute('CREATE TABLE t (a INTEGER, b TEXT)')
cur.execute("INSERT INTO t VALUES (1, 'x')")
desc = desc_full = None
for _row in cur.execute('SELECT a, b FROM t'):
    desc = cur.description
    if has_column_metadata:
        desc_full = cur.description_full
print(f"description = {desc}")
assert desc is not None
assert desc[0][:2] == ('a', 'INTEGER') and desc[1][:2] == ('b', 'TEXT')

if has_column_metadata:
    print(f"description_full = {desc_full}")
    assert desc_full[0] == ('a', 'INTEGER', 'main', 't', 'a'), desc_full
    assert desc_full[1] == ('b', 'TEXT', 'main', 't', 'b'), desc_full
else:
    print("description_full skipped: this SQLite was built without column metadata")
conn.close()

In [ ]:
# Example 4: User-defined scalar function via Connection.create_scalar_function.
import apsw

conn = apsw.Connection(':memory:')

def shouty(s: str) -> str:
    return s.upper() + '!'

conn.create_scalar_function('shouty', shouty, numargs=1, deterministic=True)

cur = conn.cursor()
result = list(cur.execute("SELECT shouty('hello'), shouty('world')"))
print(result)
assert result == [('HELLO!', 'WORLD!')]
conn.close()


In [ ]:
# Example 5: Transactions + APSW-style exception types.
import apsw

conn = apsw.Connection(':memory:')
cur = conn.cursor()
cur.execute('CREATE TABLE accts (id INTEGER PRIMARY KEY, balance INTEGER NOT NULL CHECK (balance >= 0))')
cur.execute('INSERT INTO accts VALUES (1, 100), (2, 50)')

# Successful explicit transaction via context manager.
with conn:
    cur.execute('UPDATE accts SET balance = balance - 30 WHERE id = 1')
    cur.execute('UPDATE accts SET balance = balance + 30 WHERE id = 2')

balances = dict(cur.execute('SELECT id, balance FROM accts'))
assert balances == {1: 70, 2: 80}, balances

# Failing transaction should roll back; CHECK violation triggers ConstraintError.
raised = False
try:
    with conn:
        cur.execute('UPDATE accts SET balance = balance - 999 WHERE id = 1')
except apsw.ConstraintError:
    raised = True
assert raised, 'expected ConstraintError on CHECK violation'

balances_after = dict(cur.execute('SELECT id, balance FROM accts'))
assert balances_after == {1: 70, 2: 80}, balances_after
print(f"transactions OK, rollback OK, balances = {balances_after}")
conn.close()
